In [1]:
# Imports
import csv
from datasets import load_from_disk
import numpy as np
from ml4setk.Parsing.Code.TreeSitterQuery import TreeSitterQuery
import random
import re
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import json
from transformers import StoppingCriteria, StoppingCriteriaList
import torch
import math
import time
import Levenshtein
from collections import Counter

In [4]:
model_name = "SmolLM135M"
language = "java"
tree_query = TreeSitterQuery(language=language)

method_distance = 1

c:\Users\lucaw\TUProjects\RP\ML4SE-toolkit\.venv\Lib\site-packages\tree_sitter\__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


In [81]:
def get_in_smell(code: str, spans: list[tuple[int, int]], smell_index: int = 0) -> tuple[str, str, str, str]:
    if smell_index >= len(spans):
        raise ValueError(f"Smell index {smell_index} out of bounds for spans of length {len(spans)}")
    satd_start, satd_end = spans[smell_index]
    prefix = code[:satd_start]
    target = code[satd_start:satd_end]
    suffix = code[satd_end:]
    comment_syntax = re.match(r'//\s*|/\*+\s*', target)
    if comment_syntax:
        prefix += comment_syntax.group(0)
        target = target[comment_syntax.end():]
    return prefix, target, suffix, comment_syntax.group(0)


def get_out_smell(code: str, spans: list[tuple[int, int]], smell_index: int = 0, method_distance: int = 0) -> tuple[str, str, str]:
    # Include SATD + everything up to the method declaration
    if smell_index >= len(spans):
        raise ValueError(f"Smell index {smell_index} out of bounds for spans of length {len(spans)}")
    satd_start, satd_end = spans[smell_index]
    context = code[satd_end:]
    methods = tree_query.parse(context, '(method_declaration) @method')
    if method_distance == -1 and len(methods) < 3:
        raise ValueError("Method distance -1 but less than 3 methods found")
    if method_distance < len(methods):
        method = methods[method_distance]
    else:
        raise ValueError(f"Method distance exceeds available methods, methods found: {len(methods)}, requested distance: {method_distance}")
    prefix = code[:satd_end + method[0]]
    target = method[2]
    suffix =  code[satd_end + method[1]:]
    return prefix, target, suffix

def remove_spans(content, spans):
    """
    Remove substrings from content based on the spans (start, end) indices.
    """
    cleaned_content = content
    for start, end in sorted(spans, reverse=True):
        cleaned_content = cleaned_content[:start] + cleaned_content[end:]
    return cleaned_content


def get_no_smell_comment(code: str) -> tuple[str, str, str, str]:
    comments = tree_query.parse(code, '(line_comment) @comment (block_comment) @comment')
    if not comments:
        raise ValueError("No comments found in the code")
    comment = random.choice(comments)
    prefix = code[:comment[0]]
    target = comment[2]
    suffix = code[comment[1]:]
    comment_syntax = re.match(r'//\s*|/\*+\s*', target)
    if comment_syntax:
        prefix += comment_syntax.group(0)
        target = target[comment_syntax.end():]
    return prefix, target, suffix, comment_syntax.group(0)

def get_no_smell_method(code: str) -> tuple[str, str, str]:
    methods = tree_query.parse(code, '(method_declaration) @method')
    if not methods:
        raise ValueError("No methods found in the code")
    method = random.choice(methods)
    prefix = code[:method[0]]
    target = method[2]
    suffix = code[method[1]:]
    return prefix, target, suffix

In [90]:
class StopOnSubstrings(StoppingCriteria):
    def __init__(self, stop_strings, tokenizer, start_len):
        self.stop_strings = stop_strings
        self.tokenizer = tokenizer
        self.start_len = start_len  # used to slice out only the generated part

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        # Decode only the newly generated part
        generated_ids = input_ids[0][self.start_len:]
        decoded = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        
        return any(stop in decoded for stop in self.stop_strings)
    
class StopOnMethodTreeSitter(StoppingCriteria):
    def __init__(self, tree_query: TreeSitterQuery, tokenizer, start_len):
        self.tree_query = tree_query
        self.tokenizer = tokenizer
        self.start_len = start_len
        self.open_bracket = False

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        last_token_id = input_ids[0, -1].item()
        last_token = self.tokenizer.decode([last_token_id])

        if not self.open_bracket:
            # Check if the last token is an opening brace, which indicates a multi-line method declaration
            if "{" in last_token:
                self.open_bracket = True
        
        # Only check when a closing brace is generated
        if "}" not in last_token and (self.open_bracket or ";" not in last_token):
            return False
        
        generated_ids = input_ids[0][self.start_len:]
        decoded = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        
        # Parse the generated code to check for method declarations
        methods = self.tree_query.parse(decoded, '(method_declaration) @method')

        # Only stop if a complete method is found and it ends with a closing brace
        if methods and ((decoded.strip().endswith("}") and methods[0][2].count("{") == methods[0][2].count("}")) or (not self.open_bracket and decoded.strip().endswith(";"))):
            return True
        
        return False
    
class BraceMatchStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, start_len):
        self.tokenizer = tokenizer
        self.start_len = start_len
        self.open_braces = 0
        self.close_braces = 0
        self.total_time = 0.0

    def add_time(self, start: float):
        self.total_time += (time.time() - start)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        start = time.time()
        last_token_id = input_ids[0, -1].item()
        last_token = self.tokenizer.decode([last_token_id])
        
        # Only check when a closing brace is generated
        if "}" not in last_token:
            self.add_time(start)
            return False
        
        generated_ids = input_ids[0][self.start_len:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        # Count open and close braces
        self.open_braces = text.count('{')
        self.close_braces = text.count('}')

        # Stop if enough balanced pairs are present
        if self.open_braces >= 1 and self.open_braces <= self.close_braces:
            self.add_time(start)
            return True
        self.add_time(start)
        return False
    
class SingleLineMethodStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, start_len):
        self.tokenizer = tokenizer
        self.start_len = start_len
        self.method_decl_pattern = re.compile(
            r"""^[\s@a-zA-Z0-9<>,\[\] \.@]*             # modifiers and annotations
                \b[\w<>]+\b\s+                          # return type
                \b\w+\b\s*                              # method name
                \([^\)]*\)\s*                           # parameters
                (throws\s+[^{;]+)?                      # optional throws clause
                ;                                       # ends with semicolon
            """, re.VERBOSE
        )
        self.total_time = 0.0

    def add_time(self, start: float):
        self.total_time += (time.time() - start)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        last_token_id = input_ids[0, -1].item()
        last_token = self.tokenizer.decode([last_token_id])

        # Only check when a semicolon is generated
        if ";" not in last_token:
            self.add_time(time.time())
            return False
        
        generated_ids = input_ids[0][self.start_len:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        # Check if the generated text matches the single-line method pattern
        if self.method_decl_pattern.match(text.strip()):
            self.add_time(time.time())
            return True
        self.add_time(time.time())
        return False
    
class LineRepetitionStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, start_len: int, filename: str, repeat_threshold: int = 5, min_line_length_ed: int = 20, edit_distance_threshold: int = 4):
        self.tokenizer = tokenizer
        self.start_len = start_len
        self.filename = filename
        self.repeat_threshold = repeat_threshold
        self.repeat_threshold_shortline = 20
        self.min_line_length_ed = min_line_length_ed
        self.edit_distance_threshold = edit_distance_threshold
        self.min_line_length_prefix = 40

    def is_similar(self, line1: str, line2: str) -> bool:
        return Levenshtein.distance(line1, line2) <= self.edit_distance_threshold
    
    def is_structurally_repetitive(self, prev_line: str, current_line: str) -> bool:
        # Check if one line is a prefix or suffix of another (ignoring whitespace)
        p, c = prev_line.strip(), current_line.strip()
        if len(c) < self.min_line_length_prefix or len(p) < self.min_line_length_prefix:
            return False
        if c.startswith(p) or p.startswith(c) or c.endswith(p) or p.endswith(c):
            return True
        return self.is_similar(p, c)  # Fall back to edit distance

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        # Only check when the last generated token is a newline
        last_token_id = input_ids[0, -1].item()
        last_token = self.tokenizer.decode([last_token_id])

        if "\n" not in last_token:
            return False  # Skip unless a new line was generated

        # Decode only the generated text
        generated_ids = input_ids[0][self.start_len:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        lines = [line.strip() for line in text.split("\n") if line.strip()]

        if len(lines) < self.repeat_threshold:
            return False

        last_line = lines[-1]

        # If the last line is too short, check for short line repetition
        if len(last_line) < self.min_line_length_ed:
            if len(lines) < self.repeat_threshold_shortline:
                return False
            else:
                same_count = sum(1 for line in lines[:-1] if line == last_line)
                if same_count + 1 >= self.repeat_threshold_shortline:
                    print(f"Stopping in file:{self.filename} due to short line repetition: '{last_line}' repeated {same_count + 1} times")
                    return True
            return False
        
        similar_count_ed = sum(1 for line in lines[:-1] if self.is_similar(line, last_line))
        if similar_count_ed + 1 >= self.repeat_threshold:
            print(f"Stopping in file:{self.filename} due to line repetition: '{last_line}' repeated {similar_count_ed + 1} times")
            return True
        
        similar_count_prefix = sum(1 for line in lines[:-1] if self.is_structurally_repetitive(line, last_line))
        if similar_count_prefix + 1 >= self.repeat_threshold:
            print(f"Stopping in file:{self.filename} due to structural line repetition: '{last_line}' repeated {similar_count_prefix + 1} times")
            return True
        
        return False
    
class RepetitionInLongSingleLine(StoppingCriteria):
    def __init__(self, tokenizer, start_len: int, filename: str, min_repeat_len=4, max_repeat_len=20, threshold=40):
        self.tokenizer = tokenizer
        self.start_len = start_len
        self.filename = filename
        self.min_repeat_len = min_repeat_len
        self.max_repeat_len = max_repeat_len
        self.threshold = threshold
        self.long_line_threshold = 250

    def has_repeating_hallucination(self, line: str):
        for size in range(self.min_repeat_len, self.max_repeat_len + 1):
            for i in range(0, len(line) - size * self.threshold + 1):
                unit = line[i:i+size]
                if all(line[i + j*size:i + (j+1)*size] == unit for j in range(self.threshold)):
                    return True
        return False
    
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        generated_ids = input_ids[0][self.start_len:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        lines = [line.strip() for line in text.split("\n") if line.strip()]
        if not lines:
            return False
        last_line = lines[-1]

        # Only check if the last line is long enough
        if len(last_line) > self.long_line_threshold:
            if self.has_repeating_hallucination(last_line):
                print(f"Stopping in file:{self.filename} due to hallucination in long line: '{last_line}'")
                return True
            else:
                self.long_line_threshold = self.long_line_threshold + 250  # Increase threshold for next checks
        return False

class RepetitionInSingleLineComment(StoppingCriteria):
    def __init__(self, tokenizer, start_len: int, filename: str, repeated_word_threshold: int = 30, long_digit_or_punct_threshold: int = 60,):
        self.tokenizer = tokenizer
        self.start_len = start_len
        self.filename = filename
        self.repeated_word_threshold = repeated_word_threshold
        self.long_digit_or_punct_threshold = long_digit_or_punct_threshold
        self.digits_or_delim_re = re.compile(rf"[\d.,]{{{self.long_digit_or_punct_threshold},}}")
        self.punct_re = re.compile(rf"([^\w\s])\1{{{self.long_digit_or_punct_threshold - 1},}}")
        self.repeated_number_punct_re = re.compile(rf"(?:\b\d+[\.\)]?\s+){{{self.repeated_word_threshold - 1},}}")

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        generated_ids = input_ids[0][self.start_len:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        # Only check if the generated text is long enough
        if len(text) < self.long_digit_or_punct_threshold:
            return False

        # Check 1: Long digit + punctuation sequences, Match N+ digits or digits with delimiters. EX: 1.1.1.1.1.1.1.
        # Check 2: Match N+ identical punctuation characters (excluding letters/numbers/whitespace). EX: #############
        # Check 3: Long repeated sequences of numbers or punctuation. EX: 1. 2. 3. 4. 5.
        if self.digits_or_delim_re.search(text) or self.punct_re.search(text) or self.repeated_number_punct_re.search(text):
            print(f"Stopping in file:{self.filename} due to repetition")
            return True

        # Check 4: Repeated words in the generated text
        words = text.split()
        word_counts = Counter(words)
        if any(count >= self.repeated_word_threshold for count in word_counts.values()):
            print(f"Stopping in file:{self.filename} due to repeated words: {word_counts.most_common(1)[0][0]} repeated {word_counts.most_common(1)[0][1]} times")
            return True

        return False

In [88]:
checkpoint = "HuggingFaceTB/SmolLM2-135M"
# checkpoint = "bigcode/starcoder2-3b"
device = "cuda" # cuda for GPU usage or "cpu" for CPU usage
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

output_dir_tresholds = "../../data/thresholds/mean-2std-all"
def load_threshold(filepath):
    with open(os.path.join(output_dir_tresholds, filepath), "r") as f:
        data = json.load(f)
    return data
max_tokens_comment = math.ceil(load_threshold(f"comment_stats_all_{model_name}.json")["all"]["threshold"])
max_tokens_method = math.ceil(load_threshold(f"method_stats_all_{model_name}.json")["method"]["threshold"])

max_context = model.config.max_position_embeddings
if model_name == "SmolLM135M":
    max_context = 2048
    max_tokens_method = math.ceil(load_threshold(f"method_stats_all_{model_name}.json")["method"]["mean"] + load_threshold(f"method_stats_all_{model_name}.json")["method"]["std"])
print(f"Max context size: {max_context}")
available_context_comment = max_context - max_tokens_comment
available_context_method = max_context - max_tokens_method

def generate_completion_comment(input: str, comment_syntax: str, filename: str) -> str:
    inputs = tokenizer(input, return_tensors="pt", truncation=False).to(model.device)
    # Ensure the input does not exceed the available context
    if inputs["input_ids"].shape[1] > available_context_comment:
        # Truncate the input to fit within the available context
        inputs["input_ids"] = inputs["input_ids"][:, -available_context_comment:]
        print(f"Input truncated to fit within available context, file: {filename}")

    # Calculate where the generation starts (token-wise)
    start_len = inputs["input_ids"].shape[1]

    stopping_criteria = StoppingCriteriaList()

    # Choose appropriate stop tokens
    if comment_syntax.startswith("//"):
        stop_strings = ["\n"]
        stopping_criteria.append(StopOnSubstrings(stop_strings, tokenizer, start_len))
        stopping_criteria.append(RepetitionInSingleLineComment(tokenizer, start_len, filename))
    elif comment_syntax.startswith("/*") or comment_syntax.startswith("/**"):
        stop_strings = ["*/"]
        stopping_criteria.append(StopOnSubstrings(stop_strings, tokenizer, start_len))
        stopping_criteria.append(LineRepetitionStoppingCriteria(tokenizer, start_len, filename))

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_tokens_comment,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria
        )

    generated_ids = outputs[0][start_len:]
    completion = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return completion

def generate_completion_method(input: str, filename: str) -> str:
    inputs = tokenizer(input, return_tensors="pt").to(model.device)
    # Ensure the input does not exceed the available context
    if inputs["input_ids"].shape[1] > available_context_method:
        # Truncate the input to fit within the available context
        inputs["input_ids"] = inputs["input_ids"][:, -available_context_method:]
        print(f"Input truncated to fit within available context, file: {filename}")

    # Calculate where the generation starts (token-wise)
    start_len = inputs["input_ids"].shape[1]
    
    stopping_criteria = StoppingCriteriaList([
        StopOnMethodTreeSitter(tree_query, tokenizer, start_len),
        LineRepetitionStoppingCriteria(tokenizer, start_len, filename),
        RepetitionInLongSingleLine(tokenizer, start_len, filename)
    ])

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_tokens_method,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria
        )

    generated_ids = outputs[0][start_len:]
    # Check if last token is end of text token
    if generated_ids[-1].item() == tokenizer.eos_token_id:
        print("Last token is EOS, the generation was stopped by the model for file:", filename)
    completion = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    methods = tree_query.parse(completion, '(method_declaration) @method')
    if methods:
        method_code = methods[0][2].strip()
        return method_code
    else:
        return completion

Max context size: 2048


In [55]:
# IN SMELL PREDICTION
smell_pos = 0
output_dir_gen = "../../data/codegen/test"
if not os.path.exists(output_dir_gen):
    os.makedirs(output_dir_gen)
output_file_gen = f"some-predictions-in-smell-{model_name}.csv"
with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["filename", "target", "prediction"])
    
    for file in tqdm(one_smell_ds.select(range(300)), desc="Processing files"):
        spans = decode_bitmask_string(file["bitmask_satd"])
        content = file["content"]
        filename = file.get("file_name", "unknown")

        try:
            prefix, target, comment_syntax = get_in_smell(content, spans, smell_pos)
        except ValueError as e:
                # If no smell at smell_pos, skip this file
                print(f"Skipping file {filename} due to error: {e}")
                continue

        # Validate prefix and target
        if not prefix.strip():
            continue
        if not target.strip():
            continue

        try:
            # Generate the completion
            
            pred = generate_completion_comment(prefix, comment_syntax, filename)
        except Exception as e:
            print(f"Error generating completion for {filename}: {e}")
            continue
        writer.writerow([filename, target, pred])

Processing files:   0%|          | 1/300 [00:00<02:17,  2.17it/s]

Generated completion for DefaultIndexerTest.java: '2.12.0'
Input truncated to fit within available context, file: UserEndpointTest.java


Processing files:   1%|          | 2/300 [00:00<01:51,  2.68it/s]

Generated completion for UserEndpointTest.java: "1. Get the user's permissions"


Processing files:   1%|          | 3/300 [00:01<01:56,  2.56it/s]

...

Generated completion for PreAuthLoginEndpoint.java: '"rememberMe", rememberMeRequestCache.isRememberMe(contextPath)'
Input truncated to fit within available context, file: TokenBasedRememberMeServices.java


Processing files:   7%|▋         | 20/300 [00:12<03:29,  1.34it/s]

Generated completion for TokenBasedRememberMeServices.java: 'has been authenticated.'
Generated completion for JwtProperties.java: 'Create the private key'


Processing files:   7%|▋         | 21/300 [00:14<03:09,  1.47it/s]


KeyboardInterrupt: 

In [73]:
# OUT SMELL PREDICTION
distances = [0]  # Distances to test
smell_pos = 0
for method_distance in distances:
    output_dir_gen = "../../data/codegen/test"
    if not os.path.exists(output_dir_gen):
        os.makedirs(output_dir_gen)
    output_file_gen = f"some-stopTree-predictions-out-smell-distance({method_distance})-{model_name}.csv"
    with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["filename", "target", "prediction"])
        
        for file in tqdm(one_smell_ds.select(range(100)), desc="Processing files"):
            spans = decode_bitmask_string(file["bitmask_satd"])
            content = file["content"]
            filename = file.get("file_name", "unknown")

            try:
                prefix, target = get_out_smell(content, spans, smell_pos, method_distance)
            except ValueError as e:
                # If no method found for method_distance or no smell at smell_pos, skip this file
                print(f"Skipping file {filename} due to error: {e}")
                continue

            # Validate prefix and target
            if not prefix.strip():
                continue
            if not target.strip():
                continue

            pred = generate_completion_method(prefix, filename)
            print(repr(pred))
            writer.writerow([filename, target, pred])

Processing files:   0%|          | 0/100 [00:00<?, ?it/s]

Input truncated to fit within available context, file: LSPosedBridge.java


Processing files: 100%|██████████| 100/100 [00:07<00:00, 13.41it/s]

Stopping in file:LSPosedBridge.java due to structural line repetition: '* @param {Object} args.args.args.args.args.args.args.args.args' repeated 5 times
'/**\n     * @param {Object} hooker\n     * @param {Object} callback\n     * @param {Object} ctx\n     * @param {Object} args\n     * @param {Object} args.length\n     * @param {Object} args.args\n     * @param {Object} args.args.length\n     * @param {Object} args.args.args\n     * @param {Object} args.args.args.length\n     * @param {Object} args.args.args.args\n     * @param {Object} args.args.args.args.length\n     * @param {Object} args.args.args.args.args\n     * @param {Object} args.args.args.args.args.length\n     * @param {Object} args.args.args.args.args.args\n     * @param {Object} args.args.args.args.args.args.args\n     * @param {Object} args.args.args.args.args.args.args.args\n     * @param {Object} args.args.args.args.args.args.args.args.args'


<span style="color: red"> Some bugs for stopcriteria
1. } close brace then open brace { also stops
2. normal open then close brace in comment also stops
3. Random only } close brace stops  -> maybe just smollm doing weird predicting end of file while clearing still need to close more braces
4. Random only stop without generating anything ''. -> might be because max context size 
</span>

In [ ]:
# No SMELL COMMENT PREDICTION
output_dir_gen = "../data/codegen/"
if not os.path.exists(output_dir_gen):
    os.makedirs(output_dir_gen)
output_file_gen = f"2k-predictions-no-smell-comment-{model_name}.csv"
with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["filename", "target", "prediction"])
    
    for file in tqdm(random_zero_smell_ds, desc="Processing files"):
        content = file["content"]
        filename = file.get("file_name", "unknown")

        try:
            prefix, target, comment_syntax = get_no_smell_comment(content)
        except ValueError as e:
            # If no comments found, skip this file
            print(f"Skipping file {filename} due to error: {e}")
            continue

        # Validate prefix and target
        if not prefix.strip():
            continue
        if not target.strip():
            continue

        try:
            # Generate the completion
            pred = generate_completion_comment(prefix, comment_syntax, filename)
        except Exception as e:
            print(f"Error generating completion for {filename}: {e}")
            continue
        writer.writerow([filename, target, pred])

Processing files:   1%|          | 16/2000 [00:55<1:46:05,  3.21s/it]

Skipping file AbstractCreateSuperCallResolutionExample.java due to error: No comments found in the code


Processing files:   1%|          | 23/2000 [01:09<1:28:02,  2.67s/it]

Skipping file NoticeService.java due to error: No comments found in the code


Processing files:   1%|▏         | 28/2000 [01:17<50:01,  1.52s/it]  

...

Processing files:  99%|█████████▉| 1982/2000 [1:00:23<00:31,  1.75s/it]

Skipping file Logger.java due to error: No comments found in the code


Processing files:  99%|█████████▉| 1987/2000 [1:00:36<00:25,  1.99s/it]

Skipping file ErrorCardHolder.java due to error: No comments found in the code


Processing files: 100%|██████████| 2000/2000 [1:00:58<00:00,  1.83s/it]


<span style="color: red;">Interesting for no comment there were <strong>5/20 license comments</strong> as target. These are the only comments in some of the files. It will do a <strong>horrible prediction</strong> for causal and those comments are not a <strong>good representative baseline</strong> for SATD comments.</span>

In [13]:
# No SMELL METHOD PREDICTION
total_time = 0.0
output_dir_gen = "../data/codegen/"
if not os.path.exists(output_dir_gen):
    os.makedirs(output_dir_gen)
output_file_gen = f"500-predictions-no-smell-method-{model_name}.csv"
with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["filename", "target", "prediction"])
    
    for file in tqdm(random_zero_smell_ds.select(range(500)), desc="Processing files"):
        content = file["content"]
        filename = file.get("file_name", "unknown")

        try:
            prefix, target = get_no_smell_method(content)
        except ValueError as e:
            # If no comments found, skip this file
            print(f"Skipping file {filename} due to error: {e}")
            continue

        # Validate prefix and target
        if not prefix.strip():
            continue
        if not target.strip():
            continue

        try:
            # Generate the completion
            pred = generate_completion_method(prefix, filename)
        except Exception as e:
            print(f"Error generating completion for {filename}: {e}")
            continue
        writer.writerow([filename, target, pred])


Processing files:   1%|          | 3/500 [00:18<48:47,  5.89s/it]  

Stopping due to line repetition: '@Param("task_item_resource_log_example")' repeated 5 times


Processing files:   1%|          | 4/500 [00:21<38:02,  4.60s/it]

Stopping due to line repetition: 'String s4 = "Hello, World!";' repeated 5 times
Input truncated to fit within available context, file: TreeWalkerTest.java


Processing files:   1%|          | 6/500 [00:51<1:37:22, 11.83s/it]

...

Stopping due to line repetition: 'public interface TestInterface {' repeated 5 times


Processing files:  99%|█████████▊| 493/500 [34:23<00:31,  4.49s/it]

Stopping due to line repetition: 'intent.putExtra("icon_size", mIconSize.getSize());' repeated 5 times
Skipping file Example3.java due to error: No methods found in the code


Processing files: 100%|██████████| 500/500 [34:43<00:00,  4.17s/it]

Stopping due to line repetition: 'dbViewerFragment.databaseViewerActivity.setTitle(stringBuilder.toString());' repeated 5 times


<span style="color: red;"> Interesting the stopping criterion does not includ single line method declarations. This makes the model generate more than it should.</span>

In [ ]:
# PREPROCESS OUT SMELL PREDICTION
distances = [0, -1]  # Distances to test
smell_pos = -1
for method_distance in distances:
    output_dir_gen = "../../data/codegen/test"
    if not os.path.exists(output_dir_gen):
        os.makedirs(output_dir_gen)
    output_file_gen = f"predictions-preprocess-out-smell-distance({method_distance})-{model_name}.csv"
    with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["filename", "target", "prediction"])
        
        for file in tqdm(multiple_smell_ds.select(range(10)), desc="Processing files"):
            spans = decode_bitmask_string(file["bitmask_satd"])
            content = file["content"]
            filename = file.get("file_name", "unknown")

            try:
                prefix, target, suffix = get_out_smell(content, spans, smell_pos, method_distance)
                # Clean the prefix by removing the SATD spans
                prefix = remove_spans(prefix, spans)

            except ValueError as e:
                # If no method found for method_distance or no smell at smell_pos, skip this file
                print(f"Skipping file {filename} due to error: {e}")
                continue

            # Validate prefix and target
            if not prefix.strip():
                continue
            if not target.strip():
                continue

            try:
                # Generate the completion
                pred = generate_completion_method(prefix, filename)
            except Exception as e:
                print(f"Error generating completion for {filename}: {e}")
                continue
            writer.writerow([filename, target, pred])

Processing files:   0%|          | 0/10 [00:00<?, ?it/s]

Input truncated to fit within available context, file: PluginReconciler.java


Processing files:  10%|█         | 1/10 [00:01<00:09,  1.03s/it]

Skipping file SharedApplicationContextFactory.java due to error: Method distance exceeds available methods, methods found: 0, requested distance: 0
Skipping file PluginStartedListener.java due to error: Method distance exceeds available methods, methods found: 0, requested distance: 0


Processing files:  50%|█████     | 5/10 [00:04<00:04,  1.09it/s]

...

Processing files:  80%|████████  | 8/10 [00:27<00:10,  5.08s/it]

Stopping in file:App.java due to line repetition: 'if (!isThrowableCritical(this)) {' repeated 5 times
Input truncated to fit within available context, file: RouterActivity.java


Processing files:  90%|█████████ | 9/10 [00:36<00:06,  6.43s/it]

Stopping in file:RouterActivity.java due to line repetition: 'private static final String TAG_PREFIX_SUFFIX_PREFIX_PREFIX_PREFIX_PREFIX_PREFIX_PREFIX_PREFIX = "Utils";' repeated 5 times
Input truncated to fit within available context, file: ListHelper.java


Processing files: 100%|██████████| 10/10 [00:40<00:00,  4.04s/it]


In [77]:
# Print content and spans for a specific file
for file in multiple_smell_ds.select(range(20)):
    if file["file_name"] == "Watcher.java":
        content = file["content"]
        print(content)
        spans = decode_bitmask_string(file["bitmask_satd"])
        print(spans)
        print(content[spans[0][0]:spans[0][1]])
        break

package run.halo.app.extension;

import java.util.List;
import java.util.concurrent.CopyOnWriteArrayList;
import reactor.core.Disposable;
import run.halo.app.extension.controller.Reconciler;

public interface Watcher extends Disposable {

    default void onAdd(Reconciler.Request request) {
        // Do nothing here, just for sync all on start.
    }

    default void onAdd(Extension extension) {
        // Do nothing here
    }

    default void onUpdate(Extension oldExtension, Extension newExtension) {
        // Do nothing here
    }

    default void onDelete(Extension extension) {
        // Do nothing here
    }

    default void registerDisposeHook(Runnable dispose) {
    }

    class WatcherComposite implements Watcher {

        private final List<Watcher> watchers;

        private volatile boolean disposed = false;

        private Runnable disposeHook;

        public WatcherComposite() {
            watchers = new CopyOnWriteArrayList<>();
        }

        @Override
   

In [82]:
import gzip
def load_jsonl_gz(path):
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

In [84]:
amount_method = 500
smell_index = -1  # Default to -1 for multiple out-smell cases
distances = [0, -1]  # Distances for out-smell
cases = [
    f"multi({smell_index})_out_smell_distance({distances[0]})_{amount_method}",
    f"multi({smell_index})_out_smell_distance({distances[1]})_{amount_method}",
    f"preprocessed_multi({smell_index})_out_smell_distance({distances[0]})_{amount_method}",
    f"preprocessed_multi({smell_index})_out_smell_distance({distances[1]})_{amount_method}",
]

In [ ]:
for case in cases:
    samples = load_jsonl_gz(f"../../data/input-target/v1-ds100k/{case}.jsonl.gz")
    output_dir_gen = "../../data/codegen/preliminary-results-smollm-1h-jobs"
    if not os.path.exists(output_dir_gen):
        os.makedirs(output_dir_gen)
    output_file_gen = f"{case}-{model_name}.csv"
    with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["filename", "target", "prediction"])

        for sample in tqdm(samples, desc=f"Processing {case} samples"):
            filename = sample["file_name"]
            prefix = sample["prefix"]
            target = sample["target"]
            
            try:
                # Generate the completion
                pred = generate_completion_method(prefix, filename)
            except Exception as e:
                print(f"Error generating completion for {filename}: {e}")
                continue
            writer.writerow([filename, target, pred])
            file.flush()
            os.fsync(file.fileno())
        

Processing multi(-1)_out_smell_distance(0)_500 samples:   2%|▏         | 2/100 [00:02<01:58,  1.21s/it]

Input truncated to fit within available context, file: JMHSample_10_ConstantFold.java


Processing multi(-1)_out_smell_distance(0)_500 samples:   3%|▎         | 3/100 [00:03<02:00,  1.24s/it]

Input truncated to fit within available context, file: PageSlider.java


Processing multi(-1)_out_smell_distance(0)_500 samples:   4%|▍         | 4/100 [00:04<01:59,  1.25s/it]

...

Processing multi(-1)_out_smell_distance(0)_500 samples:  98%|█████████▊| 98/100 [13:32<00:15,  7.50s/it]

Stopping in file:JMHSample_25_API_GA.java due to line repetition: 'private final String basePath4;' repeated 5 times


Processing multi(-1)_out_smell_distance(0)_500 samples:  99%|█████████▉| 99/100 [13:40<00:07,  7.80s/it]

Input truncated to fit within available context, file: NavigableTextPane.java


Processing multi(-1)_out_smell_distance(0)_500 samples: 100%|██████████| 100/100 [13:41<00:00,  8.21s/it]


In [ ]:
# PREPROCESS OUT SMELL PREDICTION


distances = [0, -1]  # Distances to test
smell_pos = -1
for method_distance in distances:
    output_dir_gen = "../../data/codegen/preliminary-results-smollm-1h-jobs"
    if not os.path.exists(output_dir_gen):
        os.makedirs(output_dir_gen)
    output_file_gen = f"predictions-preprocess-out-smell-distance({method_distance})-{model_name}.csv"
    with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["filename", "target", "prediction"])
        
        for file in tqdm(multiple_smell_ds.select(range(10)), desc="Processing files"):
            spans = decode_bitmask_string(file["bitmask_satd"])
            content = file["content"]
            filename = file.get("file_name", "unknown")

            try:
                prefix, target, suffix = get_out_smell(content, spans, smell_pos, method_distance)
                # Clean the prefix by removing the SATD spans
                prefix = remove_spans(prefix, spans)

            except ValueError as e:
                # If no method found for method_distance or no smell at smell_pos, skip this file
                print(f"Skipping file {filename} due to error: {e}")
                continue

            # Validate prefix and target
            if not prefix.strip():
                continue
            if not target.strip():
                continue

            try:
                # Generate the completion
                pred = generate_completion_method(prefix, filename)
            except Exception as e:
                print(f"Error generating completion for {filename}: {e}")
                continue
            writer.writerow([filename, target, pred])